# 01. Data preparation (party identity, preliminary model)

Builds the analysis data for the **preliminary model** from the GSS 1972–2024 cumulative file (Release 3a).

**Input**
- `data/gss7224_r3.dta` — GSS 1972–2024 Cross-Sectional Cumulative Data, Release 3a, Stata format. The data are not
  redistributed here; download them from NORC (https://gss.norc.org/get-the-data) and save the file as `data/gss7224_r3.dta`.
- `data/occ10-to-egp-class-crosswalk.csv` — 2010 Census occupation code → EGP class crosswalk (Morgan, 2017)

**Steps**
1. Merge split-ballot survey-experiment versions of the same item into one variable (`<name>_c`).
2. Keep respondents with a valid `partyid` and create the binary target `partyid_1`
   (1 = Democrat, `partyid` 0–2; 2 = Republican, `partyid` 4–6; 0 = independent/other, dropped later).
3. Recode occupations (`occ10` and parallel variables for spouse, parents, etc.) into EGP classes;
   drop legacy occupation/prestige codes and years-of-schooling variables.
4. Split by year and ballot (a/b/c), drop columns with more than 5% missing values, then drop rows with any missing value (list-wise deletion).

**Output**
- `data/cleaned_data_for_preliminary_analysis.pkl` — `{(ballot, 'XGBoost'): {year: DataFrame}}`, the input of `02_preliminary.ipynb`.

## Load data and merge survey-experiment items

In [ ]:
import pandas as pd
df_origin = pd.read_stata("data/gss7224_r3.dta", convert_categoricals=False)

# Merge survey-experiment versions of the same item
variable_groups = [
    # 1972-2018 vars
    # a
    ('amicable', 'amicably', 'amicablz'),
    # b
    ('bible', 'bibley'),
    ('blnumok', 'blnumoky'),
    # c
    ('class', 'classy'),
    ('clean', 'cleany', 'cleanz'),
    ('colaff', 'colaffy'),
    ('conarmy', 'conarmyy'),
    ('conbus', 'conbusy'),
    ('conclerg', 'conclergy'),
    ('coneduc', 'coneducy'),
    ('confed', 'confedy'),
    ('confinan', 'confinay'),
    ('conjudge', 'conjudgy'),
    ('conlabor', 'conlabory'),
    ('conlegis', 'conlegiy'),
    ('conmedic', 'conmediy'),
    ('conpress', 'conpresy'),
    ('consci', 'consciy'),
    ('consider', 'considey', 'considez'),
    ('control', 'controly', 'controlz'),
    ('contv', 'contvy'),
    ('courts', 'courtsy'),
    # d
    ('discaff', 'discaffy'),
    ('divlaw', 'divlawy'),
    # e
    ('eqwlth', 'eqwlthy'),
    # f
    ('fepol', 'fepoly'),
    # g
    ('genecanx', 'genecany'),
    ('getahead', 'getaheay'),
    ('grass', 'grassy'),
    # h
    ('hinumok', 'hinumoky'),
    ('honest', 'honesty', 'honestz'),
    # i
    ('interest', 'interesy', 'interesz'),
    # j
    ('judgment', 'judgmeny', 'judgmenz'),
    # m
    ('manners', 'mannersy', 'mannersz'),
    # n
    ('nataid', 'nataidy', 'nataidz'),
    ('natarms', 'natarmsy', 'natarmsz'),
    ('natcity', 'natcityy', 'natcityz'),
    ('natcrime', 'natcrimy', 'natcrimz'),
    ('natdrug', 'natdrugy', 'natdrugz'),
    ('nateduc', 'nateducy', 'nateducz'),
    ('natenvir', 'natenviy', 'natenviz'),
    ('natfare', 'natfarey', 'natfarez'),
    ('natheal', 'nathealy', 'nathealz'),
    ('natmass', 'natmassz'),
    ('natpark', 'natparkz'),
    ('natrace', 'natracey', 'natracez'),
    ('natroad', 'natroadz'),
    ('natsoc', 'natsocz'),
    ('natspac', 'natspacy', 'natspacz'),
    ('newsfrom', 'newsfrmy'),
    # o
    ('obeys', 'obeysy', 'obeysz'),
    # p
    ('polviews', 'polviewx', 'polviewy'),
    ('posslq', 'posslqy'),
    ('prayer', 'prayery', 'prayerx'),
    # r
    ('responsi', 'responsy', 'responsz'),
    ('role', 'roley', 'rolez'),
    # s
    ('scifrom', 'scifromy'),
    ('seeksci', 'seeksciy'),
    ('studious', 'studiouy', 'studiouz'),
    ('success', 'successy', 'successz'),
    # t
    ('trust', 'trusty'),
    # u
    ('uswar', 'uswary'),

    # 2024 vars
    # Traditional Wording Experiments
    ('marsame', 'marsamey'),
    ('posslq', 'posslqy'),
    ('racdif1', 'racdif1x'),
    ('racdif2', 'racdif2x'),
    ('racdif3', 'racdif3x'),
    ('racdif4', 'racdif4x'),
    ('racdif5', 'racdif5x'),
    ('xmovie', 'xmoviey'),

    # Volunteered Responses Experiments
    ('bible', 'biblev', 'biblenv'),
    ('courts', 'courtsv', 'courtsnv'),
    ('discaffw', 'discaffwv', 'discaffwnv'),
    ('divlaw', 'divlawv', 'divlawnv'),
    ('fair', 'fairv', 'fairnv'),
    ('fepol', 'fepolv', 'fepolnv'),
    ('fucitzn', 'fucitznv', 'fucitznnv'),
    ('getahead', 'getaheadv', 'getaheadnv'),
    ('grass', 'grassv', 'grassnv'),
    ('helpful', 'helpfulv', 'helpfulnv'),
    ('kidssol', 'kidssolv', 'kidssolnv'),
    ('postlife', 'postlifev', 'postlifenv'),
    ('prayer', 'prayerv', 'prayernv'),
    ('racopen', 'racopenv', 'racopennv'),
    ('reliten', 'relitenv', 'relitennv'),
    ('trust', 'trustv', 'trustnv'),
    ('uscitzn', 'uscitznv', 'uscitznnv'),

    # Gridded Variables Experiments
    ('abany', 'abanyg'),
    ('abdefect', 'abdefectg'),
    ('abhlth', 'abhlthg'),
    ('abnomore', 'abnomoreg'),
    ('abpoor', 'abpoorg'),
    ('abrape', 'abrapeg'),
    ('absingle', 'absingleg'),
    ('suicide1', 'suicide1g'),
    ('suicide2', 'suicide2g'),
    ('suicide4', 'suicide4g'),

    # Gender-neutral Wording Experiments
    ('colcom', 'colcomy'),
    ('famdif16', 'famdif16y'),
    ('letdie1', 'letdie1y'),
    ('libath', 'libathy'),
    ('libcom', 'libcomy'),
    ('libmslm', 'libmslmy'),
    ('librac', 'libracy'),
    ('polabuse', 'polabusey'),
    ('polattak', 'polattaky'),
    ('polhitok', 'polhitoky'),
    ('spkath', 'spkathy'),
    ('spkcom', 'spkcomy'),
    ('spkmslm', 'spkmslmy'),
    ('spkrac', 'spkracy'),

    # found later
    ('libhomo', 'libhomoy'),
    ('spkhomo', 'spkhomoy')
]

# --- Merge each group into one variable ---
df1 = df_origin.copy()
original_cols_to_drop = set() # set avoids duplicates

for group in variable_groups:
    # Keep only the group members present in the data
    existing_vars = [var for var in group if var in df1.columns]

    if not existing_vars:
        continue

    # New column name from the first variable of the group (e.g. nataid_c)
    base_name = existing_vars[0]
    combined_col = f"{base_name.lower()}_c"

    # Combine
    # Start from the first variable and fill its missing values with the other versions
    combined_series = df1[existing_vars[0]]
    for i in range(1, len(existing_vars)):
        combined_series = combined_series.combine_first(df1[existing_vars[i]])

    # Add the merged series to the data
    df1[combined_col] = combined_series

    # Mark the original columns for removal
    original_cols_to_drop.update(existing_vars)

# --- Drop the original (unmerged) variables ---
df1 = df1.drop(columns=list(original_cols_to_drop))

## Target variable, EGP class coding, and dropped variables

In [ ]:
import pandas as pd
import numpy as np

df = df1.copy()
df = df[df['partyid'].notna()]

# Target variables
# partyid
# 0 strong democrat ~ 3 independent ~ 6 strong republican, 7 other party

df = df.assign(
    partyid_1 = np.select(
        [df['partyid'].isin([0, 1, 2]), df['partyid'].isin([4, 5, 6])],
        [1, 2],
        default=0
    ),
    partyid_2 = np.select(
        [df['partyid'].isin([0, 1]), df['partyid'].isin([5, 6])],
        [1, 2],
        default=0
    ),
    partyid_3 = np.select(
        [df['partyid'].isin([0]), df['partyid'].isin([6])],
        [1, 2],
        default=0
    )
)

# Years of schooling are dropped; education is represented by degree only
eduyear = ['educ', 'speduc', 'paeduc', 'maeduc']

# Occupation codes are harmonized to the 2010 Census scheme

# Convert occ10 to EGP class
# --- Convert occ10 and parallel variables to EGP class (based on GSS MR125) ---
print("\n--- EGP class conversion: start ---")
crosswalk_file = 'data/occ10-to-egp-class-crosswalk.csv'
try:
    df_egp_crosswalk = pd.read_csv(crosswalk_file)
    print(f"Loaded '{crosswalk_file}'.")

    # Make crosswalk occ10 numeric
    df_egp_crosswalk['occ10'] = pd.to_numeric(df_egp_crosswalk['occ10'], errors='coerce')
    # Keep only the EGP 10-class code for the merge
    crosswalk_mini = df_egp_crosswalk[['occ10', 'egp10_10']].copy()

    # --- 1. Respondent (R): 10-, 11- and 12-class versions ---
    print("Processing Respondent (R)...")
    df['occ10'] = pd.to_numeric(df['occ10'], errors='coerce')
    df = pd.merge(df, crosswalk_mini, on='occ10', how='left')

    # Classes reassigned to class IV (self-employed): IIIa, IIIb, V, VI, VIIa
    egp_base_classes_for_iv = [3, 4, 8, 9, 10]

    # Respondent's EGP 11-class and 12-class
    if all(col in df.columns for col in ['wrkslf', 'numemps', 'egp10_10']):
        df['wrkslf'] = pd.to_numeric(df['wrkslf'], errors='coerce')
        df['numemps'] = pd.to_numeric(df['numemps'], errors='coerce')

        # 11-Class (egp10_11)
        df['egp10_11'] = df['egp10_10']
        condition_11 = (df['egp10_10'].isin(egp_base_classes_for_iv) & (df['wrkslf'] == 1))
        df.loc[condition_11, 'egp10_11'] = 5 # 5 = class IVab

        # 12-Class (egp10_12)
        df['egp10_12'] = df['egp10_10']
        condition_12a = (condition_11 & (df['numemps'] > 0)) # employers
        condition_12b = (condition_11 & (df['numemps'] == 0)) # own-account workers
        df.loc[condition_12a, 'egp10_12'] = 5 # 5 = class IVa
        df.loc[condition_12b, 'egp10_12'] = 6 # 6 = class IVb
        print("Respondent (R): EGP 10-, 11- and 12-class created.")
    else:
        print("Warning: respondent 'wrkslf' or 'numemps' not found; skipping 11/12-class.")

    # --- 2. Parallel variables (spouse, parents, etc.): up to 11-class ---
    # (occ_code, self_employ_code, new_egp10_col_name, new_egp11_col_name)
    individuals_to_process = {
        'sp': ('spocc10', 'spwrkslf', 'sp_egp10_10', 'sp_egp10_11'),
        'pa': ('paocc10', 'pawrkslf', 'pa_egp10_10', 'pa_egp10_11'),
        'ma': ('maocc10', 'mawrkslf', 'ma_egp10_10', 'ma_egp10_11'),
        'kd': ('kdocc10', 'kdwrkslf', 'kd_egp10_10', 'kd_egp10_11'),
        'sb': ('sbocc10', 'sbwrkslf', 'sb_egp10_10', 'sb_egp10_11'),
        'sphed': ('sphedocc10', 'sphedslf', 'sphed_egp10_10', 'sphed_egp10_11'), # spouse's parent uses 'sphedslf'
    }

    for prefix, (occ_col, slf_col, egp10_col, egp11_col) in individuals_to_process.items():
        if occ_col in df.columns:
            print(f"--- Processing {prefix} ({occ_col}) ---")

            # Harmonize type
            df[occ_col] = pd.to_numeric(df[occ_col], errors='coerce')

            # Step A: merge the 10-class code
            temp_crosswalk = crosswalk_mini.rename(columns={'occ10': occ_col, 'egp10_10': egp10_col})
            df = pd.merge(df, temp_crosswalk, on=occ_col, how='left')

            # Step B: 11-class code (when the self-employment variable 'slf_col' exists)
            if slf_col in df.columns:
                df[slf_col] = pd.to_numeric(df[slf_col], errors='coerce')
                df[egp11_col] = df[egp10_col] # copy the 10-class code

                condition_11_parallel = (
                    df[egp10_col].isin(egp_base_classes_for_iv) &
                    (df[slf_col] == 1)
                )
                df.loc[condition_11_parallel, egp11_col] = 5 # 5 = class IVab
                print(f"Created '{egp10_col}' and '{egp11_col}'.")
            else:
                print(f"Warning: '{slf_col}' not found; creating '{egp10_col}' only.")
        else:
            print(f"Skipping {prefix}: '{occ_col}' not found.")

    print("--- EGP class conversion: done ---")

except FileNotFoundError:
    print(f"Error: {crosswalk_file} not found.")
except KeyError as e:
    print(f"Error: required column not found. {e}")
except Exception as e:
    print(f"Unexpected error: {e}")
# --- End of EGP conversion ---


# Keep prestg10 and sei10; drop the other occupation codes
occupation = [
    # === 1. Respondent (R) legacy variables (1970, 1980, ISCO, DOT) ===
    'occ', 'industry', 'prestige', 'sei', 'occ80', 'indus80', 'prestg80', 'isco68', 'isco681', 'isco88',
    'dotdata', 'dotpeop', 'dotthng', 'dotged', 'dotsvp', 'dotpres', 'occfirst', 'presfrst', 'wrkslfst', 'indfirst',

    # === 2. Spouse legacy variables ===
    'spocc', 'spind', 'sppres', 'spsei', 'spocc80', 'spind80', 'sppres80',
    'spisco68', 'spisc681', 'spisco88', 'spdotdat', 'spdotpeo', 'spdotthn', 'spdotged', 'spdotsvp', 'spdotpre',

    # === 3. Parents legacy variables ===
    'paocc16', 'paind16', 'papres16', 'paocc80', 'paind80', 'papres80', 'pasei',
    'paisc681', 'paisco68', 'paisco88', 'padotdat', 'padotpeo', 'padotthn', 'padotged', 'padotsvp', 'padotrpe',
    'maocc80', 'maind80', 'mapres80', 'masei', 'maisc681', 'maisco88',

    # === 4. Other family legacy variables (children, siblings, spouse's parents) ===
    'kdocc80', 'kdind80', 'kdpres80', 'sbocc80', 'sbind80', 'sbpres80', 'sphedocc', 'sphedind', 'sphedpre',

    # === 5. Alternative prestige scales (...105PLUS) ===
    'prestg105plus', 'sppres105plus', 'papres105plus', 'mapres105plus', 'kdpres105plus', 'sbpres105plus', 'sphedpres105plus',

    # === 6. Metadata / flag variables (not used for modelling) ===
    # Flags for the existence of verbatim text
    'occindv', 'spoccindv', 'paoccindv', 'maoccindv', 'kdoccindv', 'sboccindv', 'sphedoccindv',

    # Coding status flags
    'occstatus', 'indstatus', 'spoccstatus', 'spindstatus', 'paoccstatus', 'paindstatus', 'maoccstatus',
    'maindstatus', 'kdoccstatus', 'kdindstatus', 'sboccstatus', 'sbindstatus', 'sphedoccstatus', 'sphedindstatus',

    # Coding tag flags
    'occtag', 'indtag', 'spocctag', 'spindtag', 'paocctag', 'paindtag', 'maocctag', 'maindtag',
    'kdoctag', 'kdindtag', 'sboctag', 'sbindtag', 'sphedocctag', 'sphedindtag',

    # === 7. Original ...10 codes replaced by EGP ===
    # (industry codes such as indus10 and indus07 included)
    'occ10', 'indus10', 'indus07', 'isco08',
    'spocc10', 'spind10',
    'paocc10', 'paind10',
    'maocc10', 'maind10',
    'kdocc10', 'kdind10',
    'sbocc10', 'sbind10',
    'sphedocc10', 'sphedind10',

    # EGP variables not used
    'egp10_10', 'egp_10_11'
    'spocc10', 'spwrkslf', 'sp_egp10_10',
    'paocc10', 'pawrkslf', 'pa_egp10_10',
    'maocc10', 'mawrkslf', 'ma_egp10_10',
    'kdocc10', 'kdwrkslf', 'kd_egp10_10',
    'sbocc10', 'sbwrkslf', 'sb_egp10_10',
    'sphedocc10', 'sphedslf', 'sphed_egp10_10' # spouse's parent uses 'sphedslf'
]

# Detailed Protestant denomination variable
protestant = ['others']

features_to_drop = [col for col in occupation if col in df.columns] +\
                [col for col in eduyear if col in df.columns] +\
                [col for col in protestant if col in df.columns]

df = df.drop(columns=features_to_drop, errors='ignore')

## Split by year and ballot

In [ ]:
# Dictionary of data by year
df_by_year = {year: data for year, data in df.groupby('year')} # {year: data}

for year, df_year in df_by_year.items():
    df_by_year[year] = df_year.dropna(axis=1, how='all') # drop columns that are entirely missing

# Data frames by year
#for year, df_year in df_by_year.items():
#    globals()[f'df{year}'] = df_year

# Split each year again by ballot

# 1. Ballot codes in the `ballot` column (a = 1, b = 2, c = 3)
ballot_conditions = {
    'a' : [1],
    'b': [2],
    'c': [3]
}

# 2. Container for all results
data_for_analysis = {}

# 3. Build the dictionary by ballot
for year, df in df_by_year.items():
    # Only years with a ballot column
    if 'ballot' in df.columns:
        for ballot, nums in ballot_conditions.items():
        # Keys are a, b, c
            key_name = f'{ballot}'
            # Initialize the key if needed
            if key_name not in data_for_analysis:
                data_for_analysis[key_name] = {}

            # Filter the ballot
            filtered_df = df[
                df['ballot'].isin(nums)
            ]
            filtered_df = filtered_df.dropna(axis=1, how = 'all')

            # Keep non-empty results
            if not filtered_df.empty:
                data_for_analysis[key_name][year] = filtered_df

## Drop columns with >5% missing values, then list-wise deletion

In [ ]:
# Drop columns by missing-value share and return a summary table

def clean_dataframes_with_summary(df_dict, threshold):
    """
    Clean a dictionary of data frames and return a summary of the changes.
    Columns whose missing-value share exceeds `threshold` are dropped,
    then rows with any remaining missing value are dropped.

    Returns:
        tuple: (cleaned dictionary, summary data frame)
    """
    # Deep copy so the input dictionary is not modified
    import copy
    import pandas as pd
    cleaned_dict = copy.deepcopy(df_dict)

    # 1. Records for the summary table
    shape_records = []

    # Loop over all data frames
    for key1, value1 in cleaned_dict.items():
        if isinstance(value1, dict):
            for key2, df in value1.items():
                print(f"--- Processing: {key1} - {key2} ---")

                # --- 1. Columns ---
                original_shape = df.shape

                # Missing-value share
                missing_ratio = df.isnull().sum() / len(df)

                # Columns to drop
                cols_to_drop = missing_ratio[missing_ratio > threshold].index

                if len(cols_to_drop) > 0:
                    print(f"Dropping {len(cols_to_drop)} columns with > {threshold*100:.0f}% missing:")
                    # print(list(cols_to_drop))
                    df.drop(columns=cols_to_drop, inplace=True)
                else:
                    print("No columns to drop.")

                # --- 2. Rows ---
                rows_before = df.shape[0]
                df.dropna(inplace=True)
                rows_after = df.shape[0]
                rows_dropped = rows_before - rows_after

                print(f"Dropped {rows_dropped} rows with remaining missing values.")

                # --- Summary ---
                cleaned_shape = df.shape
                print(f"Shape: {original_shape} -> {cleaned_shape}")
                print("-" * 30)

                # 2. Store this record
                record = {
                    'ballot': key1,
                    'year': key2,
                    'original_shape': original_shape,
                    'cleaned_shape': cleaned_shape,
                    'cols_dropped': len(cols_to_drop),
                    'rows_dropped': rows_dropped
                }
                shape_records.append(record)

                # Put the cleaned data frame back
                cleaned_dict[key1][key2] = df

    # 3. Records to data frame
    summary_df = pd.DataFrame(shape_records)

    # 4. Return the cleaned dictionary and the summary
    return cleaned_dict, summary_df

In [ ]:
# Missing-value threshold for dropping columns
MISSING_VALUE_THRESHOLD = 0.05

# --- Clean ---
cleaned_data_for_analysis, cleaned_data_for_analysis_shape = clean_dataframes_with_summary(data_for_analysis, MISSING_VALUE_THRESHOLD)

## Save

In [ ]:
import pickle
import os

# The model notebooks share the key structure {(ballot, model): {year: df}}.
# The preliminary model uses the full feature set for every model,
# so each ballot is stored under the XGBoost key.
data_to_save = {
    (ballot, model_name): year_dict
    for ballot, year_dict in cleaned_data_for_analysis.items()
    for model_name in ['XGBoost']
}
filename = "data/cleaned_data_for_preliminary_analysis.pkl"

with open(filename, 'wb') as file:
    pickle.dump(data_to_save, file)
print(f"Saved: '{filename}'")
print(f"Number of keys: {len(data_to_save)} (one per ballot)")